# Event-Based Framing Analysis

Analyzing how frame usage changes around major events (2015-2021).

In [ ]:
!pip install pandas numpy matplotlib seaborn pyarrow scipy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [ ]:
df = pd.read_parquet('data/merged_topic_and_frames_filtered.parquet')
df['date'] = pd.to_datetime(df['date'])
df['quarter'] = df['date'].dt.to_period('Q')
print(f'Loaded {len(df):,} articles')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')

frames = ['economic', 'fairness', 'public_op', 'political', 'quality_life', 
          'crime', 'culture', 'health', 'legality', 'morality', 
          'policy', 'regulation', 'security', 'cap&res']

frame_matrix = np.array(df['vector'].tolist())
for i, frame in enumerate(frames):
    df[f'frame_{frame}'] = frame_matrix[:, i]

frame_cols = [f'frame_{f}' for f in frames]

valence_map = {'Left': 'Left', 'Lean Left': 'Left', 'Center': 'Center', 'Lean Right': 'Right', 'Right': 'Right'}
df['valence'] = df['bias'].map(valence_map)

In [ ]:
events = [
    {'name': '2016 Election', 'quarter': '2016Q4'},
    {'name': 'Charlottesville', 'quarter': '2017Q3'},
    {'name': 'Parkland Shooting', 'quarter': '2018Q1'},
    {'name': 'COVID-19 Start', 'quarter': '2020Q1'},
    {'name': 'George Floyd', 'quarter': '2020Q2'},
    {'name': '2020 Election', 'quarter': '2020Q4'},
    {'name': 'Jan 6 Riot', 'quarter': '2021Q1'},
]

event_quarters = {e['quarter']: e['name'] for e in events}
print('Events:', list(event_quarters.values()))

## Global Frame Usage Over Time with Events

In [ ]:
quarterly_global = df.groupby('quarter')[frame_cols].mean()
quarterly_global.columns = frames
quarters_list = quarterly_global.index.astype(str).tolist()

# split into two charts for readability
frames_group1 = ['economic', 'fairness', 'public_op', 'political', 'quality_life', 'crime', 'culture']
frames_group2 = ['health', 'legality', 'morality', 'policy', 'regulation', 'security', 'cap&res']

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

for ax, frame_group, title in zip(axes, [frames_group1, frames_group2], ['Frames (Part 1)', 'Frames (Part 2)']):
    for frame in frame_group:
        ax.plot(quarters_list, quarterly_global[frame], label=frame, linewidth=1.5)
    
    for q, name in event_quarters.items():
        if q in quarters_list:
            idx = quarters_list.index(q)
            ax.axvline(x=idx, color='gray', linestyle='--', alpha=0.5)
            ax.text(idx + 0.1, ax.get_ylim()[1] * 0.9, name, rotation=90, va='top', fontsize=8)
    
    ax.set_title(f'Frame Usage Over Time - {title}')
    ax.set_xlabel('Quarter')
    ax.set_ylabel('Usage Rate')
    ax.legend(loc='upper left', ncol=4)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Frame Changes Around Each Event

In [ ]:
def frame_change_around_event(event_q, n_before=2):
    quarters_list = sorted(df['quarter'].unique())
    event_idx = quarters_list.index(pd.Period(event_q))
    
    if event_idx < n_before:
        return None
    
    before_qs = quarters_list[event_idx - n_before:event_idx]
    event_q_period = quarters_list[event_idx]
    
    before_df = df[df['quarter'].isin(before_qs)]
    event_df = df[df['quarter'] == event_q_period]
    
    before_mean = before_df[frame_cols].mean()
    event_mean = event_df[frame_cols].mean()
    
    change = event_mean - before_mean
    change.index = frames
    return change

event_changes = {}
for e in events:
    change = frame_change_around_event(e['quarter'])
    if change is not None:
        event_changes[e['name']] = change

change_df = pd.DataFrame(event_changes).T
change_df.style.format('{:+.1%}').background_gradient(cmap='PiYG', vmin=-0.1, vmax=0.1)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(change_df, annot=True, fmt='+.0%', cmap='PiYG', center=0, ax=ax)
plt.title('Frame Usage Change at Event vs 2 Quarters Before')
plt.tight_layout()
plt.show()

In [ ]:
print('Largest frame changes per event:\n')
for event_name, changes in event_changes.items():
    top_changes = changes.abs().sort_values(ascending=False).head(3)
    print(f'{event_name}:')
    for frame in top_changes.index:
        print(f'  {frame}: {changes[frame]:+.1%}')
    print()

## Frame Changes by Political Valence Around Events

In [ ]:
def frame_change_by_valence(event_q, n_before=2):
    quarters_list = sorted(df['quarter'].unique())
    event_idx = quarters_list.index(pd.Period(event_q))
    
    if event_idx < n_before:
        return None
    
    before_qs = quarters_list[event_idx - n_before:event_idx]
    event_q_period = quarters_list[event_idx]
    
    results = {}
    for valence in ['Left', 'Center', 'Right']:
        v_df = df[df['valence'] == valence]
        before_mean = v_df[v_df['quarter'].isin(before_qs)][frame_cols].mean()
        event_mean = v_df[v_df['quarter'] == event_q_period][frame_cols].mean()
        change = event_mean - before_mean
        change.index = frames
        results[valence] = change
    
    return pd.DataFrame(results)

key_events = ['George Floyd', 'COVID-19 Start', '2016 Election', 'Jan 6 Riot']
for event in events:
    if event['name'] in key_events:
        valence_change = frame_change_by_valence(event['quarter'])
        if valence_change is not None:
            print(f"\n=== {event['name']} ===")
            display(valence_change.style.format('{:+.1%}').background_gradient(cmap='PiYG', vmin=-0.1, vmax=0.1, axis=None))

In [ ]:
print('Left-Right framing divergence at events:')
print('(positive = Left increased more than Right)\n')

divergence_data = []
for event in events:
    valence_change = frame_change_by_valence(event['quarter'])
    if valence_change is not None:
        left_right_diff = valence_change['Left'] - valence_change['Right']
        top_divergence = left_right_diff.abs().sort_values(ascending=False).head(3)
        print(f"{event['name']}:")
        for frame in top_divergence.index:
            print(f'  {frame}: {left_right_diff[frame]:+.1%}')
        print()
        
        divergence_data.append({'event': event['name'], 'avg_divergence': left_right_diff.abs().mean()})

div_df = pd.DataFrame(divergence_data).sort_values('avg_divergence', ascending=False)
print('\nAverage Left-Right divergence by event:')
for _, row in div_df.iterrows():
    print(f"  {row['event']}: {row['avg_divergence']:.1%}")

## Topic-Specific Event Analysis

In [ ]:
event_topic_pairs = [
    ('George Floyd', 'crime'),
    ('COVID-19 Start', 'healthcare'),
    ('COVID-19 Start', 'pandemic'),
    ('2016 Election', 'elections and politics'),
    ('2020 Election', 'elections and politics'),
    ('Jan 6 Riot', 'elections and politics'),
]

def frame_change_for_topic(event_q, topic, n_before=2):
    quarters_list = sorted(df['quarter'].unique())
    event_idx = quarters_list.index(pd.Period(event_q))
    
    if event_idx < n_before:
        return None
    
    before_qs = quarters_list[event_idx - n_before:event_idx]
    event_q_period = quarters_list[event_idx]
    
    topic_df = df[df['topic_top1'] == topic]
    before_mean = topic_df[topic_df['quarter'].isin(before_qs)][frame_cols].mean()
    event_mean = topic_df[topic_df['quarter'] == event_q_period][frame_cols].mean()
    
    change = event_mean - before_mean
    change.index = frames
    return change

print('Frame changes for topic-event pairs:\n')
for event_name, topic in event_topic_pairs:
    event_q = next(e['quarter'] for e in events if e['name'] == event_name)
    change = frame_change_for_topic(event_q, topic)
    if change is not None:
        top_changes = change.abs().sort_values(ascending=False).head(3)
        print(f'{event_name} + {topic}:')
        for frame in top_changes.index:
            print(f'  {frame}: {change[frame]:+.1%}')
        print()